# Download Kaggle dataset

In [ ]:
pip install kaggle

In [ ]:
from google.colab import files

# 上傳 kaggle.json
files.upload()

# 創建 kaggle 資料夾並移動 API 憑證
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/

# 設定權限
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle competitions download -c boy-or-girl-2025-new
!unzip boy-or-girl-2025-new.zip -d ./dataset


Archive:  boy-or-girl-2025-new.zip
  inflating: ./dataset/Boy_or_girl_test_sandbox_sample_submission.csv  
  inflating: ./dataset/boy or girl 2025 test no ans_missingValue.csv  
  inflating: ./dataset/boy or girl 2025 train_missingValue.csv  


# Dataset

**boy or girl 2025 train_missingValue.csv** - the training set   
**boy or girl 2025 test no ans_missingValue.csv** - the test set   
**Boy_or_girl_test_sandbox_sample_submission.csv** - a sample submission file in the correct format   



In [ ]:
import os

dataset_path = "./dataset"
files = os.listdir(dataset_path)
print(files)


['Boy_or_girl_test_sandbox_sample_submission.csv', 'boy or girl 2025 train_missingValue.csv', 'boy or girl 2025 test no ans_missingValue.csv']


In [ ]:
import pandas as pd

train_file = "boy or girl 2025 train_missingValue.csv"
train = pd.read_csv(os.path.join(dataset_path, train_file))

test_file = "boy or girl 2025 test no ans_missingValue.csv"
test = pd.read_csv(os.path.join(dataset_path, test_file))

print(train)
print(test)

      id  gender star_sign phone_os  height  weight  sleepiness     iq  \
0      1       2       處女座    Apple   154.0    43.0         NaN    NaN   
1      2       2       處女座    Apple   156.0    47.0         NaN  130.0   
2      3       1       射手座      NaN   170.0    61.0         NaN   90.0   
3      4       1       射手座    Apple   170.0    62.0         4.0  100.0   
4      5       2       射手座  Android   158.0    67.0         NaN  128.0   
..   ...     ...       ...      ...     ...     ...         ...    ...   
418  419       1       處女座  Android   166.0    66.0         4.0   90.0   
419  420       1       牡羊座  Android   176.0    65.0         4.0   87.0   
420  421       1       NaN    Apple   174.0    72.0         2.0    NaN   
421  422       2       天蠍座      NaN   167.0    50.0         3.0  180.0   
422  423       1       雙魚座  Android   173.0    68.0         3.0   66.0   

     fb_friends   yt                     self_intro  
0         583.0    0                      Beautiful  
1  

In [ ]:
train = train[['id', 'gender', 'height', 'weight', 'self_intro']]
test = test[['id', 'gender', 'height', 'weight', 'self_intro']]

# 連google 雲端

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 資料前處理

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor

In [ ]:
def check_missing_values(df):
    """
    輸入一個 DataFrame，回傳含有遺漏值的欄位名稱與其遺漏值數量。
    """
    missing_series = df.isnull().sum()
    missing_features = missing_series[missing_series > 0]
    if len(missing_features) == 0:
        print("沒有任何欄位有遺漏值")
    else:
        print("以下欄位含有遺漏值 (欄位: 遺漏值數量):")
        for feature, count in missing_features.items():
            print(f"{feature}: {count}")

In [ ]:
path = "/content/drive/MyDrive/資科/Assignment 2/KNN imputation + Feature engineering/"

In [ ]:
train_missing_info = check_missing_values(train)
test_missing_info  = check_missing_values(test)

以下欄位含有遺漏值 (欄位: 遺漏值數量):
height: 74
weight: 85
self_intro: 104
以下欄位含有遺漏值 (欄位: 遺漏值數量):
height: 68
weight: 96
self_intro: 93


In [ ]:
# ------------------------------------------------
# (A) 資料前處理
# ------------------------------------------------

# A-1. 將 height、weight 的離群值視為遺漏值（使用 IQR 判定）
def replace_outliers_with_nan(df_train, df_test, cols):
    """
    以訓練集的 IQR 計算上下界，超出者在訓練與測試集中均視為 np.nan。
    """
    for col in cols:
        Q1 = df_train[col].quantile(0.25)
        Q3 = df_train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 訓練集
        df_train.loc[(df_train[col] < lower_bound) | (df_train[col] > upper_bound), col] = np.nan
        # 測試集
        df_test.loc[(df_test[col] < lower_bound) | (df_test[col] > upper_bound), col] = np.nan

replace_outliers_with_nan(train, test, ['height', 'weight'])

In [ ]:
# ------------------------------------------------
# (B) 使用 KNNImputer 進行遺漏值補值
# ------------------------------------------------
from sklearn.preprocessing import StandardScaler

# 要補值的數值欄位（不包含 id, gender）
impute_cols = ['height', 'weight']

# KNN 補值
imputer = KNNImputer(n_neighbors=8)
train[impute_cols] = imputer.fit_transform(train[impute_cols])
test[impute_cols] = imputer.transform(test[impute_cols])  # ⚠️ 用訓練集的規則


# A-8. 增加 self_intro_len 與 self_intro_words 欄位
# 先確認 self_intro 欄位存在，若可能不存在可先做檢查
if 'self_intro' in train.columns:
    train['self_intro_len'] = train['self_intro'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)
    train['self_intro_words'] = train['self_intro'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

if 'self_intro' in test.columns:
    test['self_intro_len'] = test['self_intro'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)
    test['self_intro_words'] = test['self_intro'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

# A-9. 增加self_intro_clean、keyword欄位
boy_keywords = [
    'handsome', 'man', 'cool', 'guy', 'gay', 'nerd', 'boy', 'dude', 'papa',
    '男', '帥'
]


girl_keywords = [
    'beautiful', 'cute', 'pretty', 'girl', 'beauty', 'adorable',
    '可愛', '女'
]

import re

# 清理文字：移除標點符號 + 小寫
def clean_intro(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)  # 移除標點符號
    return text.strip()

# 關鍵字分類邏輯
def keyword_category(text):
    text = clean_intro(text)
    if not text:
        return 0
    has_boy = any(kw in text for kw in boy_keywords)
    has_girl = any(kw in text for kw in girl_keywords)
    if has_girl:
        return 2
    elif has_boy:
        return 1
    else:
        return 0

# 處理訓練集
train['self_intro_clean'] = train['self_intro'].fillna('').apply(clean_intro)
train['keyword'] = train['self_intro_clean'].apply(keyword_category)

# 處理測試集
test['self_intro_clean'] = test['self_intro'].fillna('').apply(clean_intro)
test['keyword'] = test['self_intro_clean'].apply(keyword_category)

# ============== 3. 移除指定欄位 ==============
drop_cols = ['self_intro', 'self_intro_clean']
for col in drop_cols:
    if col in train.columns:
        train.drop(columns=col, inplace=True)
    if col in test.columns:
        test.drop(columns=col, inplace=True)

# Step 3: 做標準化
scale_cols = impute_cols + ['keyword'] + ["self_intro_len"]	+ ["self_intro_words"]
scaler = StandardScaler()
train[scale_cols] = scaler.fit_transform(train[scale_cols])
test[scale_cols] = scaler.transform(test[scale_cols])

In [ ]:
# ------------------------------------------------
# (C) 結果輸出
# ------------------------------------------------
print(train)
print(test)


      id  gender    height    weight  self_intro_len  self_intro_words  \
0      1       2 -2.197845 -2.045453        0.109229         -0.392232   
1      2       2 -1.929586 -1.655754        2.654892          1.961161   
2      3       1 -0.051777 -0.291806        1.763910          0.784465   
3      4       1 -0.051777 -0.194381        0.872928         -0.392232   
4      5       2 -1.661328  0.292744       -0.018054         -0.392232   
..   ...     ...       ...       ...             ...               ...   
418  419       1 -0.588294  0.195319        2.145760          3.137858   
419  420       1  0.752999  0.097894       -1.036320         -0.980581   
420  421       1  0.484740  0.779868       -1.036320         -0.980581   
421  422       2 -0.454165 -1.363479       -1.036320         -0.980581   
422  423       1  0.350611  0.390168        0.491078          0.196116   

      keyword  
0    3.473647  
1   -0.408394  
2   -0.408394  
3   -0.408394  
4   -0.408394  
..        ...  

In [ ]:
train_missing_info = check_missing_values(train)
test_missing_info  = check_missing_values(test)


沒有任何欄位有遺漏值
沒有任何欄位有遺漏值


In [ ]:
train

,id,gender,height,weight,self_intro_len,self_intro_words,keyword
0,1,2,-2.197845,-2.045453,0.109229,-0.392232,3.473647
1,2,2,-1.929586,-1.655754,2.654892,1.961161,-0.408394
2,3,1,-0.051777,-0.291806,1.763910,0.784465,-0.408394
3,4,1,-0.051777,-0.194381,0.872928,-0.392232,-0.408394
4,5,2,-1.661328,0.292744,-0.018054,-0.392232,-0.408394
...,...,...,...,...,...,...,...
418,419,1,-0.588294,0.195319,2.145760,3.137858,-0.408394
419,420,1,0.752999,0.097894,-1.036320,-0.980581,-0.408394
420,421,1,0.484740,0.779868,-1.036320,-0.980581,-0.408394
421,422,2,-0.454165,-1.363479,-1.036320,-0.980581,-0.408394


In [ ]:
test

,id,gender,height,weight,self_intro_len,self_intro_words,keyword
0,1,0,0.000101,-0.015313,-0.527187,-0.392232,-0.408394
1,2,0,0.618869,1.559267,0.109229,-0.392232,-0.408394
2,3,0,-2.063716,-1.850604,-1.036320,-0.980581,-0.408394
3,4,0,0.350611,2.046391,-1.036320,-0.980581,-0.408394
4,5,0,-0.856552,-0.681505,0.109229,0.196116,-0.408394
...,...,...,...,...,...,...,...
421,422,0,-1.393069,0.585018,0.491078,1.372813,3.473647
422,423,0,-1.124811,-0.973780,-1.036320,-0.980581,-0.408394
423,424,0,0.350611,0.195319,2.273043,1.961161,-0.408394
424,425,0,-0.722423,-1.071205,-0.527187,-0.392232,-0.408394


In [ ]:
train['id'] = train['id'].astype("int")
test['id']  = test['id'].astype("int")

# 特徵工程

In [ ]:
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from joblib import dump

# ============== 1. 複製前處理後的資料 ==============
df_train = train.copy()
df_test = test.copy()

In [ ]:
# ============== 2. 在進入 XGBoost 前，先將 gender 的 1,2 → 0,1 ==============
# 確認 'gender' 是否存在
if 'gender' in train.columns:
    df_train['gender'] = df_train['gender'].map({1: 0, 2: 1})
if 'gender' in test.columns:
    df_test['gender'] = df_test['gender'].map({1: 0, 2: 1})

In [ ]:
# ============== 4. 指定目標欄位 (y) 與特徵欄位 (X) ==============
# 'gender' 是我們要預測的目標
target_col = 'gender'

# 如果是分類問題，請確保 target_col 是可用於分類 (int/str)，且無遺漏
# 若實際情況不同，請自行調整
y = df_train[target_col]
X = df_train.drop(columns=[target_col])  # 其餘都視為特徵

# 測試集同樣分割 (之後預測時用)
X_test = df_test.drop(columns=[target_col], errors='ignore')  # 若 test 沒有 target，也不會報錯
# 若測試集其實也有 keyword，可在做最後評估比較。但此處假設實際應用場合測試集沒有真實 label


In [ ]:
# ============== 5. 使用 XGBoost + RFE 做特徵篩選 ==============
# 初始化一個 XGBoost 分類器
xgb_model = XGBClassifier(
    # scale_pos_weight=scale_pos_weight,
    n_estimators=100,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# 先把 id 從 X 中拿掉 (或任何你不想做 RFE 的欄位)
X_for_RFE = X.drop(columns=['id'], errors='ignore')

# 先用所有特徵來做一次 RFE，選出所有特徵的排名
# 這裡設定 step=1，會依次移除最不重要特徵，直到剩下 1 個
rfe = RFE(
    estimator=xgb_model,
    n_features_to_select=1,  # 這裡先把所有特徵做 ranking
    step=1
)
rfe.fit(X_for_RFE, y)

print("X.columns 長度：", len(X.columns))
print("rfe.ranking_ 長度：", len(rfe.ranking_))

# RFE 會給出 ranking_ 屬性，表示特徵重要性排名 (1 = 最重要, 2 = 次之, ...)
feature_ranks = pd.DataFrame({
    'feature': X_for_RFE.columns,
    'rank': rfe.ranking_
}).sort_values(by='rank', ascending=True)

print("==== RFE 全特徵排名 ====")
print(feature_ranks)

# 取出排名最前的 10 個特徵
top10_features = feature_ranks[feature_ranks['rank'] == 1]['feature'].tolist()
# 但如果資料很多特徵，ranking_ = 1 可能不只 10 個 (因為並列第一)
# 通常我們會依照完整排序後再切前10
# 因此要更精準一些，可以手動根據排序來取前10
feature_ranks_sorted = feature_ranks.sort_values(by='rank')
top10_features = feature_ranks_sorted.iloc[:10]['feature'].tolist()

print("\n==== 最重要特徵 ====")
print(top10_features)

X.columns 長度： 6
rfe.ranking_ 長度： 5
==== RFE 全特徵排名 ====
            feature  rank
0            height     1
4           keyword     2
1            weight     3
3  self_intro_words     4
2    self_intro_len     5

==== 最重要特徵 ====
['height', 'keyword', 'weight', 'self_intro_words', 'self_intro_len']


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

In [ ]:
# ============== 6. 依照 top 1~5 特徵，做交叉驗證，找出最佳組合 ==============
# 我們會用 StratifiedKFold 做分類的交叉驗證
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 設定要測試的特徵數量列表
feature_counts = [1, 2, 3, 4, 5]

best_score = -np.inf
best_num_features = None
best_model = None

for k in feature_counts:
    # 取前 k 個特徵
    selected_features = top10_features[:k]

    # 初始化模型
    model = XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        enable_categorical=True
    )

    # 交叉驗證 (這裡用 accuracy 做範例，可依需求改為 f1, roc_auc 等)
    cv_scores = cross_val_score(model, X[selected_features], y,
                                cv=skf, scoring='accuracy')
    mean_score = np.mean(cv_scores)
    print(f"使用前 {k} 個特徵，CV 平均分數: {mean_score:.4f}")

    # 比較是否更好
    if mean_score > best_score:
        best_score = mean_score
        best_num_features = k
        best_model = model

# 在找到最佳的 k 後，用該數量的特徵重新訓練最佳模型
selected_features = top10_features[:best_num_features]
best_model.fit(X[selected_features], y)  # 訓練

print(f"\n==== 最佳特徵數量: {best_num_features}，CV 平均: {best_score:.4f} ====")
print(f"特徵: {selected_features}")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

使用前 1 個特徵，CV 平均分數: 0.8486


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

使用前 2 個特徵，CV 平均分數: 0.8888


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

...

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

使用前 4 個特徵，CV 平均分數: 0.8793


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

使用前 5 個特徵，CV 平均分數: 0.8677

==== 最佳特徵數量: 2，CV 平均: 0.8888 ====
特徵: ['height', 'keyword']


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# ============== 7. 儲存最佳模型 (選擇性) ==============
dump(best_model, path + 'best_xgb_model(All)_best_KNN=8.joblib')
print("最佳模型已儲存為 best_xgb_model(All)_best_KNN=8.joblib")

最佳模型已儲存為 best_xgb_model(All)_best_KNN=8.joblib


In [ ]:
# ============== 8. 使用最佳模型預測測試集 ==============
# 注意：測試集必須有相同的特徵處理
# 先確保 selected_features 在 X_test 中存在
X_test_selected = X_test[selected_features]

# 預測結果 (若為分類問題，可用 predict_proba 或 predict)
test_pred = best_model.predict(X_test_selected) + 1

# 假設想看機率，可用 predict_proba
test_pred_proba = best_model.predict_proba(X_test_selected)[:, 1]


In [ ]:
# ============== 9. 輸出或儲存預測結果 (自行調整) ==============
test['pred'] = test_pred
test['pred_prob'] = test_pred_proba

# 儲存預測結果
submission = pd.DataFrame({
    'id': test['id'],
    'gender': test['pred']
})

submission.to_csv(path + 'kaggle_submit_best.csv', index=False)
print("已將預測結果輸出到 kaggle_submit_best.csv")

submission[['gender']].value_counts()


已將預測結果輸出到 kaggle_submit_best.csv


,count
gender,
1,303
2,123


In [ ]:
# 取前 50% 的資料列
half_submission = submission.iloc[:len(submission) // 2]

# 對 gender 欄位計數
gender_counts_half = half_submission[['gender']].value_counts()

# 顯示結果
print(gender_counts_half)

gender
1         153
2          60
Name: count, dtype: int64
